# ===============================
# 0. Install dependencies
# ===============================


In [53]:
# !pip install openai  
# !pip install faiss-cpu
# !pip install pandas 
# !pip install numpy 
# !pip install tqdm
# !pip install python-dotenv

import os
import pandas as pd
import numpy as np
import sqlite3
import dotenv
from openai import OpenAI
from tqdm import tqdm
import re
import faiss
from pathlib import Path
from dotenv import load_dotenv

# Laad de .env-variabelen
load_dotenv()

# Haal de API key op
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✅ API key gevonden!")
else:
    print("❌ Geen API key gevonden. Controleer of .env bestaat en de juiste sleutel bevat.")

client = OpenAI(api_key=api_key) 


✅ API key gevonden!



# ===============================
# 1. Load CSVs
# ===============================

In [63]:
# 1. Load CSVs with metadata
DATA_DIR = "text_input_metadata"
all_files = sorted(Path(DATA_DIR).glob("*.csv"))

dfs = []
for f in all_files:
    # Extract inv_nr as everything before the first underscore (can include digits and letters)
    inv_nr = f.stem.split("_")[0]
    df_temp = pd.read_csv(f, sep=",")
    # Only keep doc pages (TANAP_ID not empty)
    df_temp = df_temp[df_temp["TANAP_ID"].notna() & (df_temp["TANAP_ID"] != "")]
    # Standardize columns
    df_temp = df_temp.rename(columns={
        "text": "text",
        "filename": "filename",
        "TANAP_ID": "tanap_id",
        "TANAP_Boundaries": "tanap_boundaries",
        "DATUM": "datum",
        "PLAATS": "plaats",
        "BESCHRIJVING": "beschrijving",
        "GLOBALISE_document_category": "doc_category"
    })
    df_temp["inv_nr"] = inv_nr
    dfs.append(df_temp)

df_all = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(df_all)} doc pages from {len(all_files)} files")


Loaded 17351 doc pages from 20 files


# ===============================
# 2. Optional: spelling normalization
# ===============================

In [64]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("„", '"').replace("’", "'")
    text = re.sub(r"[:;]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

APPLY_NORMALIZATION = True

if APPLY_NORMALIZATION:
    df_all["text"] = df_all["text"].apply(normalize_text)



# ===============================
# 3. Split text into overlapping chunks
# ===============================

In [68]:
# 3. Split text into chunks (do NOT cross doc boundaries)
CHUNK_SIZE = 300
CHUNK_OVERLAP = 75

def chunk_within_docs(df_inv, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    all_chunks = []
    for tanap_id, df_doc in df_inv.groupby("tanap_id"):
        # Metadata for this doc (take first row)
        meta = df_doc.iloc[0]
        all_words = []
        page_boundaries = []
        word_count = 0
        for idx, row in df_doc.iterrows():
            words = str(row["text"]).split()
            all_words.extend(words)
            page_boundaries.append((word_count, row["filename"]))
            word_count += len(words)
        start = 0
        while start < len(all_words):
            end = min(start + chunk_size, len(all_words))
            chunk_words = all_words[start:end]
            pages_in_chunk = [p for idx_p, p in page_boundaries if start <= idx_p < end]
            if not pages_in_chunk:
                pages_in_chunk = [df_doc.iloc[-1]["filename"]]
            all_chunks.append({
                "inv_nr": meta["inv_nr"],
                "tanap_id": tanap_id,
                "chunk_id": start // (chunk_size - overlap),
                "text": " ".join(chunk_words),
                "start_page": pages_in_chunk[0],
                "pages": pages_in_chunk,
                "datum": meta["datum"],
                "plaats": meta["plaats"],
                "beschrijving": meta["beschrijving"],
                "doc_category": meta["doc_category"]
            })
            start += chunk_size - overlap
    return all_chunks

all_chunks_data = []
for inv_nr in df_all['inv_nr'].unique():
    df_inv = df_all[df_all['inv_nr'] == inv_nr].copy()
    all_chunks_data.extend(chunk_within_docs(df_inv))

print(f"Created {len(all_chunks_data)} chunks across {df_all['inv_nr'].nunique()} inventory numbers")


Created 18252 chunks across 20 inventory numbers


In [69]:
# print(chunk_meta[:3])  # voorbeeld metadata
# This variable does not exist anymore; remove or replace with a preview of all_chunks_data:
print(all_chunks_data[:3])  # voorbeeld metadata

[{'inv_nr': '1120', 'tanap_id': 5084.0, 'chunk_id': 0, 'text': '756877"17"6 79813"7"- ontfangen Cargasoenen restanten in Taijditan versonden Car" Aen d Ede H=rs Bewinthebberen E. dr Erntfeste wijse voorsienige Ter Camer tot amsterdam ende zeer discrete heeren Onsen Jongsten aen VEde was in dato 23=en octobr Ao 1635 waerbij vEde doenmael hebben aengecundicht den opulenten toevoer, van alderhande Chineese waeren, ende dat sich den handel in saijouant zoo lancx zoo beter begon te vertoonen, mitsgaders de goede hoope ende apparentie dieder was om die van Mattanw en soulangh — hunnen hoochmoet te dempen, ende het mnorm faijct ten tijde van - d\' Hr Nuijts aen d\'onse begaen te revengieren, Item wat Cargasoenen voorleden zuijder Mouson naer Japan ende int voorste vant Noorder Mouson naer Bata versonden waren te weten. Ao 1635 adij 19 Augustij en 20=en ditto pr de scheepen Nieuw amsterdam wassenaer ende Groe naer Japan een Cargasoes monteren te samen .. .. T _ adij 29=en ditto Pr T Jacht venh


# ===============================
# 4. Store chunks + metadata in SQLite
# ===============================

In [70]:
# 4. Store chunks + metadata in SQLite (with extra columns)
DB_FILE = "text-metadata-sqlite/voc_documents.db"
conn = sqlite3.connect(DB_FILE)
cur = conn.cursor()

# Drop and recreate the table to ensure schema matches (fixes missing tanap_id column)
cur.execute("DROP TABLE IF EXISTS documents")
cur.execute("""
CREATE TABLE documents (
    inv_nr TEXT,
    tanap_id TEXT,
    start_page TEXT,
    pages TEXT,
    chunk_id INTEGER,
    datum TEXT,
    plaats TEXT,
    beschrijving TEXT,
    doc_category TEXT,
    text TEXT
)
""")
conn.commit()

# Insert all chunks with correct metadata
for chunk_data in all_chunks_data:
    pages_str = ";".join(chunk_data['pages']) if chunk_data['pages'] else ""
    cur.execute("""
        INSERT INTO documents (inv_nr, tanap_id, start_page, pages, chunk_id, datum, plaats, beschrijving, doc_category, text)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            str(chunk_data['inv_nr']),
            str(chunk_data['tanap_id']),
            str(chunk_data['start_page']),
            pages_str,
            int(chunk_data['chunk_id']),
            str(chunk_data['datum']),
            str(chunk_data['plaats']),
            str(chunk_data['beschrijving']),
            str(chunk_data['doc_category']),
            str(chunk_data['text'])
        ))

conn.commit()
conn.close()
print(f"✅ Inserted {len(all_chunks_data)} chunks into database")

✅ Inserted 18252 chunks into database


# ===============================
# 5. FAISS helpers per inv_nr
# ===============================

In [77]:
EMB_DIR = Path("embeddings")
EMB_DIR.mkdir(exist_ok=True)

def embeddings_file(inv_nr):
    return EMB_DIR / f"{inv_nr}.index"

def load_or_create_embeddings(inv_nr):
    """
    Load existing FAISS embeddings for an inventory number,
    or create them (including metadata with configurable weights)
    if not available.
    """

    # ====== ⚖️ WEIGHTS — adjust these to tune importance ======
    WEIGHT_PLAATS = 2.0
    WEIGHT_VESTIGING = 2.0
    WEIGHT_JAAR = 1.5
    WEIGHT_BESCHRIJVING = 0.5
    WEIGHT_TEXT = 1.0
    # ==========================================================

    def repeat_weighted(text, weight):
        n = max(1, int(round(weight)))
        return (" " + text) * n

    def expand_places(raw_str):
        if not raw_str or not isinstance(raw_str, str):
            return ""
        parts = [p.strip() for p in raw_str.split("|") if p.strip()]
        return " ".join([f"Plaats: {p}." for p in parts]) if parts else ""

    def safe_col(df, col):
        """Always return a Series (string type), even if column missing."""
        if col in df.columns:
            return df[col].fillna("").astype(str)
        else:
            return pd.Series([""] * len(df), index=df.index, dtype=str)

    db_path = Path("text-metadata-sqlite/voc_documents.db")
    emb_file = Path(f"embeddings/{inv_nr}.index")

    # --- Load chunks from SQLite ---
    with sqlite3.connect(db_path) as conn:
        df_chunks = pd.read_sql_query(
            f"SELECT * FROM documents WHERE inv_nr='{inv_nr}' ORDER BY chunk_id",
            conn
        )

    if df_chunks.empty:
        raise ValueError(f"⚠️ Geen data gevonden voor inv_nr {inv_nr} in de database.")

    # --- Safe column loading ---
    df_chunks["text"] = safe_col(df_chunks, "text")
    df_chunks["plaats"] = safe_col(df_chunks, "plaats")
    df_chunks["vestiging"] = safe_col(df_chunks, "vestiging")
    df_chunks["beschrijving"] = safe_col(df_chunks, "beschrijving")
    df_chunks["datum"] = safe_col(df_chunks, "datum")

    # Extract only the year (yyyy)
    df_chunks["jaar"] = df_chunks["datum"].str.extract(r"(\d{4})")[0].fillna("")

    # --- Combine weighted metadata into a single text field ---
    df_chunks["combined_text"] = (
        repeat_weighted(df_chunks["plaats"].apply(expand_places), WEIGHT_PLAATS) +
        repeat_weighted("Vestiging: " + df_chunks["vestiging"] + ".", WEIGHT_VESTIGING) +
        repeat_weighted("Jaar: " + df_chunks["jaar"] + ".", WEIGHT_JAAR) +
        repeat_weighted("Beschrijving: " + df_chunks["beschrijving"] + ".", WEIGHT_BESCHRIJVING) +
        repeat_weighted(df_chunks["text"], WEIGHT_TEXT)
    )

    # Remove empty or whitespace-only entries
    df_chunks = df_chunks[df_chunks["combined_text"].str.strip() != ""]
    chunks_inv = df_chunks["combined_text"].tolist()

    if not chunks_inv:
        raise ValueError(f"⚠️ Alle chunks voor {inv_nr} zijn leeg na opschonen.")

    # --- Load or create embeddings ---
    if emb_file.exists():
        faiss_idx = faiss.read_index(str(emb_file))
        print(f"✅ Loaded FAISS index for {inv_nr}")
    else:
        print(f"🧠 Computing embeddings for {inv_nr} ({len(chunks_inv)} chunks, with weighted metadata)...")

        embeddings = []
        batch_size = 50
        for i in tqdm(range(0, len(chunks_inv), batch_size)):
            batch = chunks_inv[i:i+batch_size]

            try:
                resp = client.embeddings.create(
                    model="text-embedding-3-large",
                    input=batch
                )
                batch_embeddings = [r.embedding for r in resp.data if hasattr(r, "embedding")]
                embeddings.extend(batch_embeddings)
            except Exception as e:
                print(f"⚠️ Fout bij batch {i}: {e}")
                continue

        if not embeddings:
            raise RuntimeError(f"❌ Geen embeddings aangemaakt voor {inv_nr}.")

        embeddings = np.array(embeddings, dtype="float32")

        dim = embeddings.shape[1]
        faiss_idx = faiss.IndexFlatL2(dim)
        faiss_idx.add(embeddings)

        emb_file.parent.mkdir(parents=True, exist_ok=True)
        faiss.write_index(faiss_idx, str(emb_file))
        print(f"💾 Saved FAISS index for {inv_nr} -> {emb_file}")

    return faiss_idx, chunks_inv



# ===============================
# 6. Query multiple inv_nrs
# ===============================

In [78]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

def is_dutch(text):
    """Detect if the text is Dutch using a simple heuristic (can be improved)."""
    # Very basic: check for common Dutch words
    dutch_words = ["de", "het", "een", "en", "van", "op", "in", "is", "niet", "dat", "met"]
    text_lower = text.lower()
    return any(word in text_lower.split() for word in dutch_words)

def translate_to_dutch(text):
    """Vertaal invoer (in eender welke taal) naar modern Nederlands."""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Je bent een ervaren vertaler gespecialiseerd in geschiedenis en (vroeg-)koloniale terminologie."},
            {"role": "user", "content": f"Vertaal dit naar modern Nederlands, behoud betekenis en historische termen: {text}"}
        ],
        temperature=0.3,
    )
    return resp.choices[0].message.content.strip()

def search_query(query_text, inv_nrs, top_k=None, translate_if_not_dutch=True):
    """
    Voert een semantische zoekopdracht uit over 1 of meer inventarisnummers.
    Haalt bijbehorende paginametadata op uit SQLite.
    """

    # --- optioneel vertalen ---
    if translate_if_not_dutch and not is_dutch(query_text):
        translated_query = translate_to_dutch(query_text)
        print(f"🌍 Oorspronkelijke query: {query_text}")
        print(f"🇳🇱 Vertaalde query: {translated_query}")
    else:
        translated_query = query_text
        print(f"🌍 Query (geen vertaling nodig): {query_text}")

    all_results = []
    results_dir = Path("results")
    results_dir.mkdir(exist_ok=True)

    for inv_nr in inv_nrs:
        print(f"\n🔎 Searching in {inv_nr} ...")

        # Laad FAISS-index en chunks
        faiss_idx, chunks_inv = load_or_create_embeddings(inv_nr)
        if not chunks_inv:
            print(f"⚠️ Geen chunks gevonden voor {inv_nr}, overslaan.")
            continue

        # Query embedding
        q_emb = np.array(
            client.embeddings.create(model="text-embedding-3-large", input=translated_query).data[0].embedding,
            dtype="float32"
        ).reshape(1, -1)

        # *** FIX: Ensure top_k doesn't exceed available chunks ***
        if top_k is None:
            top_k = len(chunks_inv)
        top_k = min(top_k, len(chunks_inv), faiss_idx.ntotal)

        # FAISS search
        D, I = faiss_idx.search(q_emb, top_k)
        scores = 1 / (1 + D.flatten())
        I = I.flatten()

        # --- Metadata ophalen uit SQLite
        with sqlite3.connect(DB_FILE) as conn:
            df_meta = pd.read_sql_query(
                f"SELECT inv_nr, tanap_id, start_page, pages, chunk_id, datum, plaats, beschrijving, doc_category, text FROM documents WHERE inv_nr='{inv_nr}' ORDER BY chunk_id",
                conn
            )

        # *** FIX: Filter out invalid indices ***
        valid_mask = I < len(df_meta)
        I = I[valid_mask]
        scores = scores[valid_mask]
        
        if len(I) == 0:
            print(f"⚠️ Geen geldige resultaten voor {inv_nr}")
            continue

        # --- Combineer chunks met resultaten
        df_res = df_meta.iloc[I].copy()
        df_res["similarity"] = scores

        # --- Opslaan
        safe_query = "".join([c for c in query_text if c.isalnum() or c in "-_ "]).strip().replace(" ", "_")
        out_file = results_dir / f"{safe_query}-{inv_nr}.csv"
        df_res.to_csv(out_file, index=False)
        print(f"💾 Resultaten opgeslagen in: {out_file}")

        all_results.append(df_res)

    # Combineer alle resultaten
    if all_results:
        df_all = pd.concat(all_results, ignore_index=True)
        return df_all[[
            "inv_nr", "tanap_id", "start_page", "pages", "chunk_id", "datum", "plaats", "beschrijving", "doc_category", "similarity", "text"
        ]]
    else:
        print("⚠️ Geen resultaten gevonden.")
        return pd.DataFrame(columns=[
            "inv_nr", "tanap_id", "start_page", "pages", "chunk_id", "datum", "plaats", "beschrijving", "doc_category", "similarity", "text"
        ])

# ===============================
# 7. Usage
# ===============================

In [79]:
query_text = "leven van een slaaf"
df_results = search_query(query_text, inv_nrs=["8023"])

# If no embeddings file exists for the given inv_nr, ensure the embeddings are saved with the correct extension.
# FAISS expects .index, but your code uses .db for embeddings. Fix this in load_or_create_embeddings:

# In the load_or_create_embeddings function, replace:
# emb_file = Path(f"embeddings/{inv_nr}.db")
# with:
# emb_file = Path(f"embeddings/{inv_nr}.index")

# So, update the function as follows:
def load_or_create_embeddings(inv_nr):
    # ...existing code...
    db_path = Path("text-metadata-sqlite/voc_documents.db")
    emb_file = Path(f"embeddings/{inv_nr}.index")  # <-- fix extension

    # ...existing code...
    # Controleer of embeddings al bestaan
    if emb_file.exists():
        faiss_idx = faiss.read_index(str(emb_file))
        print(f"✅ Loaded FAISS index for {inv_nr}")
    else:
        print(f"🧠 Computing embeddings for {inv_nr} ({len(chunks_inv)} chunks)...")
        # ...existing code...
        faiss.write_index(faiss_idx, str(emb_file))  # <-- will now save as .index
    return faiss_idx, chunks_inv

🌍 Query (geen vertaling nodig): leven van een slaaf

🔎 Searching in 8023 ...
🧠 Computing embeddings for 8023 (134 chunks, with weighted metadata)...


100%|██████████| 3/3 [00:02<00:00,  1.22it/s]


💾 Saved FAISS index for 8023 -> embeddings/8023.index
💾 Resultaten opgeslagen in: results/leven_van_een_slaaf-8023.csv


## BM25 Index Creation (Keyword Search)

BM25 indices enable hybrid search combining semantic embeddings with exact keyword matching. This is useful for finding specific names, places, and technical terms.
Create BM25 indices for all inventory numbers in the corpus.

In [ ]:
import pickle
import sqlite3
from rank_bm25 import BM25Okapi
from pathlib import Path
from tqdm import tqdm

# Get list of all inventory numbers in database
db_path = Path("text-metadata-sqlite/voc_documents.db")
bm25_dir = Path("bm25_indices")
bm25_dir.mkdir(exist_ok=True)

with sqlite3.connect(db_path) as conn:
    df_inv = pd.read_sql_query("SELECT DISTINCT inv_nr FROM documents ORDER BY inv_nr", conn)
    inv_nrs = df_inv['inv_nr'].tolist()

print(f"Creating BM25 indices for {len(inv_nrs)} inventory numbers...")

# Create BM25 index for each inventory number
for inv_nr in tqdm(inv_nrs, desc="BM25 Index Creation"):
    bm25_file = bm25_dir / f"{inv_nr}.pkl"
    
    # Skip if already exists
    if bm25_file.exists():
        continue
    
    # Load all chunks for this inventory
    with sqlite3.connect(db_path) as conn:
        df_chunks = pd.read_sql_query(
            "SELECT text FROM documents WHERE inv_nr=? ORDER BY chunk_id",
            conn,
            params=(inv_nr,)
        )
    
    if df_chunks.empty:
        print(f"⚠️ No data for {inv_nr}")
        continue
    
    # Prepare texts and tokenize
    texts = df_chunks['text'].fillna("").astype(str).tolist()
    tokenized_corpus = [text.lower().split() for text in texts]
    
    # Create BM25 index
    bm25 = BM25Okapi(tokenized_corpus)
    
    # Save to pickle file
    with open(bm25_file, 'wb') as f:
        pickle.dump({'bm25': bm25, 'texts': texts}, f)

print(f"✅ BM25 indices created in {bm25_dir}/")

## Testing Hybrid Search

Test the hybrid semantic + BM25 search on a sample query:

In [ ]:
from app_core import search_query, get_openai_client

# Test on a sample inventory number
test_inv = inv_nrs[0] if inv_nrs else "1120"

# Example queries
test_queries = [
    "Jan Pieterszoon Coen",  # Name (BM25 excels)
    "Banda eilanden",  # Place (BM25 excels)
    "volcanic eruptions and their consequences",  # Conceptual (semantic excels)
]

client = get_openai_client()

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    # Search with semantic only
    df_semantic = search_query(
        query_text=query,
        inv_nrs=[test_inv],
        top_k=5,
        translate_if_not_dutch=True,
        use_bm25=False,
    )
    
    print(f"\n🔵 Semantic search (BM25 disabled):")
    print(df_semantic[['inv_nr', 'similarity', 'semantic_score', 'start_page', 'text']].head(3).to_string())
    
    # Search with hybrid
    df_hybrid = search_query(
        query_text=query,
        inv_nrs=[test_inv],
        top_k=5,
        translate_if_not_dutch=True,
        use_bm25=True,
        bm25_weight=0.3,
    )
    
    print(f"\n🟣 Hybrid search (BM25 weight=0.3):")
    print(df_hybrid[['inv_nr', 'similarity', 'semantic_score', 'bm25_score', 'start_page', 'text']].head(3).to_string())